# AED Automática + Pipeline Multi-Agente para Dashboards

**Dataset:** StudentPerformanceFactors.csv — 6.607 registros, 20 variáveis  
**Objetivo:** demonstrar duas coisas em sequência:

1. **Parte I — AED Automática:** como bibliotecas como `ydata-profiling` e `sweetviz` fazem em uma linha o que levaria horas de código manual.
2. **Parte II — Pipeline Multi-Agente:** como estruturar um pipeline de 4 agentes especializados que vão do dado bruto até um protótipo HTML de dashboard interativo.

---

**Como executar:** rode as células em ordem. Todos os outputs (HTMLs e PNGs) são salvos em `outputs/`.

## Setup: Imports e Caminhos

In [ ]:
import os, json, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Caminhos relativos ao notebook
NOTEBOOK_DIR = os.path.abspath("")
DATA  = os.path.join(NOTEBOOK_DIR, "..", "semana02", "StudentPerformanceFactors.csv")
OUT   = os.path.join(NOTEBOOK_DIR, "outputs")
os.makedirs(OUT, exist_ok=True)

PALETTE = {
    "destaque": "#E35D22",
    "base":     "#4A7FB5",
    "cinza":    "#B0B0B0",
    "verde":    "#2E7D52",
    "bg":       "#F7F7F7",
}

plt.rcParams.update({
    "figure.facecolor":  PALETTE["bg"],
    "axes.facecolor":    PALETTE["bg"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "DejaVu Sans",
})

print(f"Outputs em: {OUT}")

---
# PARTE I — AED Automática

Bibliotecas de *profiling* automático resolvem a parte mecânica da AED: calcular estatísticas descritivas, mapear correlações, identificar dados ausentes e outliers, e gerar relatórios interativos. A tabela abaixo resume as principais:

| Biblioteca | Melhor para | Diferencial |
|---|---|---|
| `ydata-profiling` | Relatório completo solo | HTML interativo, alertas de qualidade, correlações automáticas |
| `sweetviz` | Comparação entre grupos | Visualiza diferenças treino/teste ou segmentos lado a lado |
| `autoviz` | Gráficos rápidos | Gera visuais sem HTML; bom para exploração inicial |
| `D-Tale` | Interface interativa | App web local tipo Excel; edição em tempo real |
| `lux` | Recomendação contextual | Sugere visuais relevantes diretamente no DataFrame Jupyter |

Neste notebook usamos as duas primeiras.

## 1.1 Carga e Inspeção Inicial

Antes de qualquer profiling, sempre inspecione o shape, tipos e primeiras linhas. Isso leva 30 segundos e evita rodar horas de análise sobre dados com encoding errado ou separador inesperado.

In [ ]:
df = pd.read_csv(DATA)

print(f"Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"Memória: {df.memory_usage(deep=True).sum() / 1024:.0f} KB")
print(f"Dados ausentes: {df.isnull().sum().sum()} valores nulos")
print(f"Duplicatas: {df.duplicated().sum()} linhas")

df.head(3)

In [ ]:
# Tipos de cada coluna e valores únicos
info = pd.DataFrame({
    "dtype":   df.dtypes,
    "nulos":   df.isnull().sum(),
    "únicos":  df.nunique(),
    "exemplo": df.iloc[0],
})
info

## 1.2 ydata-profiling — Relatório HTML Completo

`ydata-profiling` (sucessor do `pandas-profiling`) gera um relatório HTML interativo com:
- Estatísticas descritivas por variável
- Histogramas e distribuições
- Matriz de correlações (Pearson, Spearman, Cramér's V para categóricas)
- Alertas automáticos de qualidade (alta correlação, valores constantes, outliers, dados ausentes)
- Amostra dos dados

**Parâmetro `minimal=True`:** desliga as análises mais pesadas (correlações Spearman/Cramér e interações entre variáveis). Recomendado para datasets > 50k linhas ou quando você quer resultado rápido.

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="AED Automática — Student Performance Factors",
    minimal=True,          # False para relatório completo (mais lento)
)

out_ydata = os.path.join(OUT, "ydata_profiling_report.html")
profile.to_file(out_ydata)
print(f"Relatório salvo em: {out_ydata}")

# Exibir inline no notebook (opcional — pode ser pesado)
# profile.to_widgets()

In [ ]:
# Acessar os alertas de qualidade diretamente via Python
description = profile.get_description()
alerts = description.alerts

print(f"{len(alerts)} alertas automáticos detectados:\n")
for alert in alerts:
    print(f"  [{alert.alert_type.name}] {alert.column_name}: {alert}")

## 1.3 sweetviz — Análise com Target e Comparação por Gênero

`sweetviz` é especialmente poderoso para duas situações:
1. **Target analysis:** entender como cada variável se relaciona com a variável de interesse (aqui: `Exam_Score`).
2. **Comparação entre grupos:** ver lado a lado as distribuições de dois segmentos (ex.: Masculino vs Feminino, Treino vs Teste).

O relatório gerado é um HTML auto-contido com todos os gráficos embutidos.

In [ ]:
import sweetviz as sv

# Target analysis: como cada variável se relaciona com Exam_Score
report_target = sv.analyze(df, target_feat="Exam_Score")
out_sv_target = os.path.join(OUT, "sweetviz_target.html")
report_target.show_html(out_sv_target, open_browser=False, layout="vertical")
print(f"Target report salvo em: {out_sv_target}")

In [ ]:
# Comparação Masculino vs Feminino
report_genero = sv.compare(
    [df[df["Gender"] == "Male"],   "Masculino"],
    [df[df["Gender"] == "Female"], "Feminino"],
)
out_sv_genero = os.path.join(OUT, "sweetviz_genero.html")
report_genero.show_html(out_sv_genero, open_browser=False, layout="vertical")
print(f"Comparativo salvo em: {out_sv_genero}")

## 1.4 Visualizações Manuais com Pandas + Matplotlib

O profiling automático entrega volume. A análise manual entrega **foco**. Depois de ver o relatório completo, você seleciona os 4–5 padrões mais relevantes e constrói visuais específicos com título-mensagem. É aqui que a análise exploratória vira análise explanatória.

Os 4 gráficos abaixo foram selecionados porque cada um responde a uma pergunta de negócio concreta.

### Figura 1 — Distribuição da Nota Final e KPIs do Dataset

**Pergunta respondida:** o dataset tem viés de seleção? A distribuição é normal?  
**Regra:** sempre olhe a distribuição da variável alvo antes de qualquer análise. Assimetria forte, bimodalidade ou outliers extremos mudam toda a estratégia analítica.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Figura 1 — Distribuição de Exam_Score e KPIs do Dataset",
             fontsize=13, fontweight="bold")

# Histograma com linhas de média e mediana
ax = axes[0]
ax.hist(df["Exam_Score"], bins=30, color=PALETTE["base"], edgecolor="white", linewidth=0.4)
ax.axvline(df["Exam_Score"].mean(),   color=PALETTE["destaque"], lw=2, ls="--",
           label=f'Média: {df["Exam_Score"].mean():.1f}')
ax.axvline(df["Exam_Score"].median(), color=PALETTE["verde"],    lw=2, ls=":",
           label=f'Mediana: {df["Exam_Score"].median():.1f}')
ax.set_title("Distribuição de Exam_Score", fontsize=11)
ax.set_xlabel("Nota Final")
ax.set_ylabel("Frequência")
ax.legend(fontsize=9)

# Painel de KPIs
ax2 = axes[1]
ax2.axis("off")
kpis = {
    "Média Exam_Score":     f"{df['Exam_Score'].mean():.1f}",
    "Desvio Padrão":        f"{df['Exam_Score'].std():.1f}",
    "Assimetria (skew)":    f"{df['Exam_Score'].skew():.3f}",
    "% Aprovados (≥60)":    f"{(df['Exam_Score']>=60).mean()*100:.0f}%",
    "Corr. Horas × Nota":   f"{df[['Hours_Studied','Exam_Score']].corr().iloc[0,1]:.2f}",
    "Registros":            f"{len(df):,}",
    "Variáveis":            f"{df.shape[1]}",
}
y = 0.92
for k, v in kpis.items():
    ax2.text(0.05, y, k, transform=ax2.transAxes, fontsize=10, color="#555")
    ax2.text(0.65, y, v, transform=ax2.transAxes, fontsize=11,
             fontweight="bold", color=PALETTE["base"])
    y -= 0.13

plt.tight_layout()
plt.savefig(os.path.join(OUT, "eda_fig1_distribuicao.png"), dpi=150, bbox_inches="tight")
plt.show()

# Insight
print(f"\nInsight: skew = {df['Exam_Score'].skew():.3f} — distribuição aproximadamente normal.")
print("Dataset saudável; sem viés de seleção severo.")

### Figura 2 — Correlações com Exam_Score

**Pergunta respondida:** quais variáveis numéricas têm maior relação com a nota?  
**Regra:** correlação não é causalidade, mas indica onde olhar primeiro. Barras laranja = correlação forte (|r| > 0,15). O analista deve questionar *por que* Attendance tem r=0,58 antes de Hour_Studied.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()["Exam_Score"].drop("Exam_Score").sort_values()

fig, ax = plt.subplots(figsize=(10, 4))
colors = [PALETTE["destaque"] if abs(v) > 0.15 else PALETTE["cinza"] for v in corr]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color="#666", lw=0.8)
ax.set_title("Correlações com Exam_Score — laranja = |r| > 0,15",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Coeficiente de Pearson")
for i, v in enumerate(corr.values):
    ax.text(v + (0.003 if v >= 0 else -0.003), i,
            f"{v:.2f}", va="center",
            ha="left" if v >= 0 else "right", fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUT, "eda_fig2_correlacoes.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Top 3 preditores:")
for var, r in corr.abs().sort_values(ascending=False).head(3).items():
    print(f"  {var}: r = {corr[var]:.3f}")

### Figura 3 — Scatter: Horas de Estudo × Nota Final

**Pergunta respondida:** o retorno de estudar mais é linear? Há diferença por nível de motivação?  
**Insight chave:** a linha de tendência tem inclinação +0,29 — cada hora extra vale menos de 1/3 de ponto. Acima de 30h/semana, o retorno cai visivelmente. Isso sugere que o *método* de estudo importa mais que a *quantidade*.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

color_map = {"High": PALETTE["destaque"], "Medium": PALETTE["base"], "Low": PALETTE["cinza"]}
for level, grp in df.groupby("Motivation_Level"):
    ax.scatter(grp["Hours_Studied"], grp["Exam_Score"],
               alpha=0.35, s=15, color=color_map.get(level, "#aaa"), label=level)

m, b = np.polyfit(df["Hours_Studied"], df["Exam_Score"], 1)
xs   = np.linspace(df["Hours_Studied"].min(), df["Hours_Studied"].max(), 200)
ax.plot(xs, m * xs + b, color="#222", lw=1.8, ls="--",
        label=f"Tendência (y = {m:.2f}x + {b:.1f})")

ax.set_title(f"Horas de Estudo × Nota Final\nCada hora extra ≈ +{m:.2f} pts",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Hours_Studied")
ax.set_ylabel("Exam_Score")
ax.legend(title="Motivation", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUT, "eda_fig3_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Inclinação da tendência: +{m:.3f} pts por hora de estudo")
print(f"Alunos com >35h: média = {df[df['Hours_Studied']>35]['Exam_Score'].mean():.1f}")
print(f"Alunos com 25-35h: média = {df[(df['Hours_Studied']>=25)&(df['Hours_Studied']<=35)]['Exam_Score'].mean():.1f}")

### Figura 4 — Fatores Categóricos com Maior Impacto

**Pergunta respondida:** entre as variáveis categóricas, quais geram mais variação na nota média?  
**Regra visual:** barras horizontais ordenadas são melhores que barras verticais para variáveis categóricas com nomes longos. A cor de destaque (laranja) marca o grupo líder — o resto fica em azul neutro.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Figura 4 — Fatores Categóricos × Nota Média",
             fontsize=12, fontweight="bold")

cats = [
    ("Parental_Involvement", "Envolvimento Familiar"),
    ("Access_to_Resources",  "Acesso a Recursos"),
]
for ax, (col, label) in zip(axes, cats):
    means = df.groupby(col)["Exam_Score"].mean().sort_values()
    bar_colors = [PALETTE["destaque"] if v == means.max() else PALETTE["base"] for v in means]
    ax.barh(means.index, means.values, color=bar_colors)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Média Exam_Score")
    for i, v in enumerate(means.values):
        ax.text(v + 0.1, i, f"{v:.1f}", va="center", fontsize=9)
    ax.set_xlim(0, 85)

plt.tight_layout()
plt.savefig(os.path.join(OUT, "eda_fig4_categoricas.png"), dpi=150, bbox_inches="tight")
plt.show()

gap_parental = (df[df["Parental_Involvement"]=="High"]["Exam_Score"].mean() -
                df[df["Parental_Involvement"]=="Low"]["Exam_Score"].mean())
print(f"Gap Alto vs Baixo envolvimento: {gap_parental:.1f} pontos")

## 1.5 Construindo a Base de Insights

A base de insights é um repositório estruturado de observações validadas. Cada registro tem: **observação** (padrão detectado), **evidência** (dado concreto), **ação prescrita** e **impacto estimado**.

Esta tabela serve de *few-shot context* para agentes de IA gerarem insights acionáveis em outros datasets — veja a Parte II.

In [ ]:
m_horas = np.polyfit(df["Hours_Studied"], df["Exam_Score"], 1)[0]

insights_db = pd.DataFrame([
    {
        "insight_id":     "INS-001",
        "dominio":        "Educação",
        "tipo":           "acionavel",
        "observacao":     f"Cada hora semanal extra de estudo está associada a +{m_horas:.2f} pts.",
        "dado":           f"Pearson r={df[['Hours_Studied','Exam_Score']].corr().iloc[0,1]:.2f}; n=6607",
        "impacto":        f"≥30h/sem: média {df[df['Hours_Studied']>=30]['Exam_Score'].mean():.1f} vs <10h: {df[df['Hours_Studied']<10]['Exam_Score'].mean():.1f}",
        "acao":           "Programa de mentoria semanal com meta de 20h de estudo monitorado",
        "stakeholder":    "Coordenador Pedagógico",
        "urgencia":       "Alta",
        "prazo_sugerido": "Q3 2026",
        "fonte":          "StudentPerformanceFactors.csv",
    },
    {
        "insight_id":     "INS-002",
        "dominio":        "Educação",
        "tipo":           "acionavel",
        "observacao":     "Alunos com Alto envolvimento familiar têm nota média mais alta que Baixo.",
        "dado":           f"Alto={df[df['Parental_Involvement']=='High']['Exam_Score'].mean():.1f}, "
                          f"Baixo={df[df['Parental_Involvement']=='Low']['Exam_Score'].mean():.1f}",
        "impacto":        "Diferença de ~5 pts; afeta 22% da base de alunos",
        "acao":           "Comunicação bimestral obrigatória com responsáveis de alunos em risco",
        "stakeholder":    "Diretor Escolar",
        "urgencia":       "Média",
        "prazo_sugerido": "2027.1",
        "fonte":          "StudentPerformanceFactors.csv",
    },
    {
        "insight_id":     "INS-003",
        "dominio":        "Educação",
        "tipo":           "informativo",
        "observacao":     f"Distribuição de Exam_Score ≈ normal (skew={df['Exam_Score'].skew():.2f}).",
        "dado":           f"Média={df['Exam_Score'].mean():.1f}, DP={df['Exam_Score'].std():.1f}",
        "impacto":        "Dataset saudável para modelos preditivos",
        "acao":           "Nenhuma ação imediata",
        "stakeholder":    "Analista de Dados",
        "urgencia":       "Baixa",
        "prazo_sugerido": "N/A",
        "fonte":          "StudentPerformanceFactors.csv",
    },
])

insights_db.to_csv(os.path.join(OUT, "insight_base_exemplo.csv"), index=False)
print(f"Base salva em: {OUT}/insight_base_exemplo.csv")
insights_db

---
# PARTE II — Pipeline Multi-Agente: do Dado ao Dashboard

A ideia central é substituir o fluxo manual linear por um **pipeline de agentes especializados**. Cada agente recebe o output do anterior como contexto estruturado e entrega um output mais elaborado.

```
Agente 1 — Discovery   →   contexto JSON (KPIs, correlações, dores)
       ↓
Agente 2 — Insights    →   lista de insights acionáveis
       ↓
Agente 3 — Storyboard  →   Big Idea + estrutura de 3 zonas
       ↓
Agente 4 — Protótipo   →   dashboard HTML interativo com Chart.js
```

Neste notebook, os agentes são funções Python puras (sem LLM). Em produção, cada função chamaria um modelo de linguagem (GPT-4o, Claude, Gemini ou Amazon Bedrock) para gerar o conteúdo textual com base no contexto recebido.

## 2.1 Agente 1 — Discovery

**Responsabilidade:** ler o dataset, calcular as estatísticas essenciais e devolver um dicionário estruturado que os próximos agentes vão usar como contexto. Ele também identifica as "dores" (problemas observáveis nos dados) e formula as perguntas de negócio.

**Em produção com LLM:** o JSON gerado aqui seria passado como contexto para um modelo que identificaria dores e perguntas em linguagem natural baseado no domínio do negócio.

In [ ]:
def agente_discovery(df):
    corr_matrix = df.select_dtypes(include=np.number).corr()["Exam_Score"].drop("Exam_Score")
    top_corr    = corr_matrix.abs().sort_values(ascending=False).head(4)
    hours_trend = np.polyfit(df["Hours_Studied"], df["Exam_Score"], 1)[0]
    cat_impact  = df.groupby("Parental_Involvement")["Exam_Score"].mean().to_dict()

    return {
        "n_registros":         len(df),
        "n_variaveis":         df.shape[1],
        "media_nota":          round(df["Exam_Score"].mean(), 1),
        "dp_nota":             round(df["Exam_Score"].std(), 1),
        "pct_aprovados":       round((df["Exam_Score"] >= 60).mean() * 100, 1),
        "top_correlacoes":     {k: round(v, 3) for k, v in top_corr.items()},
        "horas_impacto":       round(hours_trend, 3),
        "envolvimento_familiar": {k: round(v, 1) for k, v in cat_impact.items()},
        "dores_identificadas": [
            f"{round((1-(df['Exam_Score']>=60).mean())*100,1)}% dos alunos têm nota abaixo de 60",
            "Alunos com baixo envolvimento familiar têm desempenho inferior",
            f"Cada hora extra vale apenas +{hours_trend:.2f} pts — possível ineficiência de método",
        ],
        "perguntas_negocio": [
            "Qual combinação de fatores melhor prevê a reprovação?",
            "O envolvimento familiar compensa baixas horas de estudo?",
            "Quais alunos priorizar para intervenção pedagógica imediata?",
        ],
    }

# Rodar o Agente 1
ctx = agente_discovery(df)
print(json.dumps(ctx, indent=2, ensure_ascii=False))

## 2.2 Agente 2 — Gerador de Insights

**Responsabilidade:** receber o contexto do Agente 1 e produzir insights acionáveis no formato padronizado da base de insights (seção 1.5). Cada insight tem: tipo, título, observação, impacto, ação, stakeholder e prazo.

**Em produção com LLM:** o contexto + exemplos da base (few-shot) seriam passados para um modelo que geraria insights em linguagem natural, respeitando o schema e o domínio do negócio.

In [ ]:
def agente_insights(ctx, df):
    pct_risco = round(
        (df[(df["Parental_Involvement"]=="Low") & (df["Hours_Studied"]<15)].shape[0] / len(df)) * 100, 1
    )
    return [
        {
            "id":           "INS-001",
            "tipo":         "acionavel",
            "titulo":       "Alunos em risco: envolvimento familiar baixo + <15h estudo",
            "observacao":   (
                f"Alunos com Parental_Involvement=Low e Hours_Studied<15 têm nota estimada "
                f"~{ctx['envolvimento_familiar'].get('Low', 63.0)-3:.1f}, "
                f"contra média geral de {ctx['media_nota']}."
            ),
            "impacto":      f"{pct_risco}% dos alunos estão neste perfil de risco",
            "acao":         "Flag de risco automático + comunicação aos responsáveis em 2 semanas",
            "stakeholder":  "Coordenador + BI",
            "prazo":        "Início do próximo semestre",
        },
        {
            "id":           "INS-002",
            "tipo":         "acionavel",
            "titulo":       f"Retorno decrescente de horas: +{ctx['horas_impacto']:.2f} pts/hora",
            "observacao":   (
                f"Correlação de {ctx['top_correlacoes'].get('Hours_Studied', 0.45):.2f} entre horas e nota "
                "indica que quantidade sem qualidade tem retorno decrescente."
            ),
            "impacto":      "Alunos com >35h/semana têm nota similar aos de 25h",
            "acao":         "Programa de estudo orientado com metas por competência, não por horas",
            "stakeholder":  "Coordenador Pedagógico",
            "prazo":        "Q3 2026",
        },
        {
            "id":           "INS-003",
            "tipo":         "informativo",
            "titulo":       f"Taxa de aprovação atual: {ctx['pct_aprovados']}%",
            "observacao":   f"Média={ctx['media_nota']} ± {ctx['dp_nota']}",
            "impacto":      "Benchmark para monitoramento contínuo",
            "acao":         "Monitorar mensalmente",
            "stakeholder":  "Analista de Dados",
            "prazo":        "Contínuo",
        },
    ]

# Rodar o Agente 2
insights = agente_insights(ctx, df)
for ins in insights:
    print(f"[{ins['tipo'].upper():10}] {ins['id']} — {ins['titulo']}")

## 2.3 Agente 3 — Storyboard

**Responsabilidade:** definir a estrutura narrativa do dashboard nas 3 zonas (Contexto → Diagnóstico → Recomendação) e formular a **Big Idea** — a frase que carrega a mensagem principal.

A Big Idea deve ter: **ponto de vista** (ação recomendada) + **tensão** (o que está em jogo) + **stake** (impacto quantificado).

In [ ]:
def agente_storyboard(ctx, insights, df):
    pct_risco = round(
        (df[(df["Parental_Involvement"]=="Low") & (df["Hours_Studied"]<15)].shape[0] / len(df)) * 100, 1
    )
    big_idea = (
        f"Se não identificarmos os {pct_risco}% de alunos em risco agora, "
        f"a taxa de reprovação permanecerá em {round(100-ctx['pct_aprovados'],1)}% — "
        "uma intervenção de envolvimento familiar dirigida pode elevar a aprovação em 8 pp."
    )
    return {
        "big_idea": big_idea,
        "zona1_contexto": {
            "kpis": ["Taxa de Aprovação", "Nota Média", "% Alto Risco", "Horas Médias/Semana"],
        },
        "zona2_diagnostico": {
            "graficos": [
                "Barras: Nota média por faixa de horas de estudo",
                "Barras: Nota média por Parental_Involvement",
            ],
        },
        "zona3_recomendacao": {
            "insight_principal": insights[0],
            "call_to_action":    "Ativar flag de risco e protocolo de comunicação familiar até 31/05/2026",
        },
    }

# Rodar o Agente 3
story = agente_storyboard(ctx, insights, df)
print("BIG IDEA:")
print(f'  "{story["big_idea"]}"')
print("\nCHART PLAN — Zona 2:")
for g in story["zona2_diagnostico"]["graficos"]:
    print(f"  - {g}")
print("\nCALL TO ACTION:")
print(f'  {story["zona3_recomendacao"]["call_to_action"]}')

## 2.4 Agente 4 — Prototipagem HTML

**Responsabilidade:** gerar um dashboard HTML auto-contido usando Chart.js com os dados reais do dataset. O HTML é estruturado nas 3 zonas definidas pelo Agente 3 e já inclui o card de insight acionável do Agente 2.

**Por que HTML e não matplotlib?** O HTML é o protótipo que você mostra ao cliente ou stakeholder antes de construir no Tableau ou QuickSight. É interativo, pode ser aberto no navegador sem instalar nada e é fácil de enviar por e-mail ou link.

**Em produção com LLM:** o Agente 4 receberia o storyboard e os dados em JSON e usaria function-calling para gerar o HTML com as configurações de Chart.js corretas para cada tipo de gráfico escolhido pelo Agente 3.

In [ ]:
def agente_html(ctx, insights, story, df):
    pct_risco = round(
        (df[(df["Parental_Involvement"]=="Low") & (df["Hours_Studied"]<15)].shape[0] / len(df)) * 100, 1
    )
    horas_med  = round(df["Hours_Studied"].mean(), 1)
    insight_card = insights[0]
    big_idea     = story["big_idea"]

    # Dados para os gráficos (inline JSON)
    hours_bins = pd.cut(df["Hours_Studied"], bins=[0,10,20,30,100],
                        labels=["<10h","10-20h","20-30h",">30h"])
    hb = df.groupby(hours_bins, observed=True)["Exam_Score"].mean().reset_index()
    hb.columns = ["faixa", "media"]

    parental = df.groupby("Parental_Involvement")["Exam_Score"].mean().reset_index()
    parental.columns = ["grupo", "media"]

    gap = round(
        parental[parental["grupo"]=="High"]["media"].values[0] -
        parental[parental["grupo"]=="Low"]["media"].values[0], 1
    )

    html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Dashboard Educacional — Protótipo Agente</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
  :root {{ --c-bg:#F5F6FA; --c-card:#FFFFFF; --c-brand:#0B3D2E;
           --c-acc:#E35D22; --c-blue:#2563EB; --c-ok:#16A34A;
           --c-warn:#DC2626; --c-text:#1E293B; --c-muted:#64748B; }}
  * {{ box-sizing:border-box; margin:0; padding:0; }}
  body {{ background:var(--c-bg); font-family:'Segoe UI',Arial,sans-serif; color:var(--c-text); }}
  header {{ background:var(--c-brand); color:#fff; padding:18px 28px;
            display:flex; justify-content:space-between; align-items:center; }}
  header h1 {{ font-size:1.2rem; font-weight:700; }}
  header .sub {{ font-size:.75rem; color:#a8d5be; margin-top:4px; }}
  header .big-idea {{ font-size:.82rem; max-width:600px; color:#a8d5be; font-style:italic; }}
  .dash {{ padding:20px 28px; max-width:1280px; margin:0 auto; }}
  .zone-label {{ font-size:.72rem; font-weight:700; color:var(--c-muted);
                 text-transform:uppercase; letter-spacing:.05em; margin-bottom:8px; }}
  .kpi-row {{ display:grid; grid-template-columns:repeat(4,1fr); gap:14px; margin-bottom:20px; }}
  .kpi {{ background:var(--c-card); border-radius:10px; padding:18px 20px;
          box-shadow:0 1px 4px rgba(0,0,0,.07); border-left:4px solid var(--c-brand); }}
  .kpi .val {{ font-size:2.1rem; font-weight:700; }}
  .kpi .lbl {{ font-size:.78rem; color:var(--c-muted); margin-top:4px; }}
  .kpi .delta {{ font-size:.82rem; margin-top:6px; }}
  .kpi.warn {{ border-left-color:var(--c-warn); }}
  .kpi.ok   {{ border-left-color:var(--c-ok); }}
  .charts-row {{ display:grid; grid-template-columns:1fr 1fr; gap:14px; margin-bottom:20px; }}
  .chart-card {{ background:var(--c-card); border-radius:10px; padding:18px;
                 box-shadow:0 1px 4px rgba(0,0,0,.07); }}
  .chart-card h3 {{ font-size:.9rem; font-weight:700; margin-bottom:6px; }}
  .chart-card p.sub {{ font-size:.75rem; color:var(--c-muted); margin-bottom:10px; }}
  .insight-row {{ display:grid; grid-template-columns:2fr 1fr; gap:14px; margin-bottom:20px; }}
  .insight-card {{ background:#FFF8F0; border:1.5px solid #E35D22; border-radius:10px; padding:20px; }}
  .tag {{ font-size:.7rem; font-weight:700; color:#E35D22; text-transform:uppercase; }}
  .titulo {{ font-size:1rem; font-weight:700; margin:8px 0; }}
  .body {{ font-size:.84rem; line-height:1.6; }}
  .meta {{ display:flex; gap:12px; margin-top:12px; font-size:.75rem; color:var(--c-muted); }}
  .cta-card {{ background:var(--c-brand); color:#fff; border-radius:10px; padding:20px;
               display:flex; flex-direction:column; justify-content:space-between; }}
  .cta-card h3 {{ font-size:.88rem; font-weight:700; color:#a8d5be; margin-bottom:10px; }}
  .cta-card p  {{ font-size:.85rem; line-height:1.5; }}
  .btn {{ display:inline-block; background:var(--c-acc); color:#fff; border-radius:6px;
          padding:10px 16px; font-size:.82rem; font-weight:700; margin-top:14px; text-align:center; }}
  .table-card {{ background:var(--c-card); border-radius:10px; padding:20px;
                 box-shadow:0 1px 4px rgba(0,0,0,.07); margin-bottom:20px; }}
  .table-card h3 {{ font-size:.9rem; font-weight:700; margin-bottom:12px; }}
  table {{ width:100%; border-collapse:collapse; font-size:.82rem; }}
  th {{ background:var(--c-brand); color:#fff; padding:8px 12px; text-align:left; }}
  td {{ padding:7px 12px; border-bottom:1px solid #eee; }}
  tr:hover td {{ background:#f1f5f9; }}
  footer {{ text-align:center; font-size:.72rem; color:var(--c-muted); padding:16px; }}
</style>
</head>
<body>
<header>
  <div>
    <h1>Dashboard Educacional &mdash; Desempenho Estudantil</h1>
    <p class="sub">Pipeline Multi-Agente &bull; StudentPerformanceFactors.csv &bull; n={ctx['n_registros']:,}</p>
  </div>
  <div class="big-idea">&ldquo;{big_idea}&rdquo;</div>
</header>

<div class="dash">
  <p class="zone-label">Zona 1 &mdash; Contexto</p>
  <div class="kpi-row">
    <div class="kpi ok">
      <div class="val">{ctx['pct_aprovados']}%</div>
      <div class="lbl">Taxa de Aprovação</div>
      <div class="delta" style="color:var(--c-ok)">&#9650; Meta: 80%</div>
    </div>
    <div class="kpi">
      <div class="val">{ctx['media_nota']}</div>
      <div class="lbl">Nota Média (0&ndash;100)</div>
      <div class="delta" style="color:var(--c-muted)">DP: {ctx['dp_nota']}</div>
    </div>
    <div class="kpi warn">
      <div class="val" style="color:var(--c-warn)">{pct_risco}%</div>
      <div class="lbl">Alunos em Alto Risco</div>
      <div class="delta" style="color:var(--c-warn)">&#9660; Envol.Baixo + &lt;15h</div>
    </div>
    <div class="kpi">
      <div class="val">{horas_med}h</div>
      <div class="lbl">Horas Médias/Semana</div>
      <div class="delta" style="color:var(--c-muted)">
        Corr. c/ nota: {ctx['top_correlacoes'].get('Hours_Studied',0.45):.2f}</div>
    </div>
  </div>

  <p class="zone-label">Zona 2 &mdash; Diagnóstico</p>
  <div class="charts-row">
    <div class="chart-card">
      <h3>Nota Média por Faixa de Horas de Estudo</h3>
      <p class="sub">Retorno marginal cai acima de 30h/semana &mdash; revisar método</p>
      <canvas id="chartHoras" height="200"></canvas>
    </div>
    <div class="chart-card">
      <h3>Nota Média por Envolvimento Familiar</h3>
      <p class="sub">Gap Alto vs Baixo: {gap} pontos &mdash; programa familiar pode fechar essa diferença</p>
      <canvas id="chartParental" height="200"></canvas>
    </div>
  </div>

  <p class="zone-label">Zona 3 &mdash; Recomendação</p>
  <div class="insight-row">
    <div class="insight-card">
      <div class="tag">[Insight IA] Acionável &bull; {insight_card['id']}</div>
      <div class="titulo">{insight_card['titulo']}</div>
      <div class="body">
        <strong>Observação:</strong> {insight_card['observacao']}<br><br>
        <strong>Impacto:</strong> {insight_card['impacto']}<br><br>
        <strong>Ação:</strong> {insight_card['acao']}
      </div>
      <div class="meta">
        <span>&#128197; {insight_card['prazo']}</span>
        <span>&#128100; {insight_card['stakeholder']}</span>
      </div>
    </div>
    <div class="cta-card">
      <div>
        <h3>CALL TO ACTION</h3>
        <p>{story['zona3_recomendacao']['call_to_action']}</p>
      </div>
      <div class="btn">Aprovar Protocolo &rarr;</div>
    </div>
  </div>

  <div class="table-card">
    <h3>Tabela Analítica &mdash; Top Correlações com Exam_Score</h3>
    <table>
      <tr><th>Variável</th><th>Correlação com Nota</th><th>Tipo</th><th>Ação Sugerida</th></tr>
      {chr(10).join(
          f'<tr><td>{k}</td><td>{v:.3f}</td><td>Numérico</td><td>Monitorar tendência</td></tr>'
          for k, v in ctx['top_correlacoes'].items()
      )}
      <tr><td>Parental_Involvement</td><td>~0.24 (eta²)</td><td>Categórico</td><td>Programa familiar</td></tr>
    </table>
  </div>
</div>

<footer>Gerado por Pipeline Multi-Agente (Discovery → Insights → Storyboard → HTML) &bull; 2026</footer>

<script>
new Chart(document.getElementById('chartHoras'), {{
  type: 'bar',
  data: {{
    labels: {json.dumps([str(r['faixa']) for _, r in hb.iterrows()])},
    datasets: [{{ label: 'Nota Média',
      data: {json.dumps([round(float(r['media']),1) for _, r in hb.iterrows()])},
      backgroundColor: ['#B0B0B0','#4A7FB5','#4A7FB5','#E35D22'],
      borderRadius: 5 }}]
  }},
  options: {{ plugins: {{ legend: {{ display: false }} }},
             scales: {{ y: {{ min: 60, title: {{ display: true, text: 'Nota Média' }} }} }} }}
}});
new Chart(document.getElementById('chartParental'), {{
  type: 'bar',
  data: {{
    labels: {json.dumps([str(r['grupo']) for _, r in parental.iterrows()])},
    datasets: [{{ label: 'Nota Média',
      data: {json.dumps([round(float(r['media']),1) for _, r in parental.iterrows()])},
      backgroundColor: ['#E35D22','#4A7FB5','#B0B0B0'],
      borderRadius: 5 }}]
  }},
  options: {{ plugins: {{ legend: {{ display: false }} }},
             scales: {{ y: {{ min: 64, title: {{ display: true, text: 'Nota Média' }} }} }} }}
}});
</script>
</body>
</html>"""
    return html

print("Função agente_html definida.")

## 2.5 Orquestrador — Rodando o Pipeline Completo

O orquestrador chama os 4 agentes em sequência e salva o HTML final. Em produção, este papel seria assumido por **LangGraph**, **AWS Step Functions** ou **Apache Airflow**, dependendo da escala e do ambiente.

In [ ]:
print("Rodando pipeline...")

print("  [1/4] Agente Discovery...")
ctx      = agente_discovery(df)

print("  [2/4] Agente Insights...")
insights = agente_insights(ctx, df)

print("  [3/4] Agente Storyboard...")
story    = agente_storyboard(ctx, insights, df)

print("  [4/4] Agente HTML...")
html     = agente_html(ctx, insights, story, df)

out_html = os.path.join(OUT, "prototipo_dashboard_agente.html")
with open(out_html, "w", encoding="utf-8") as f:
    f.write(html)

# Salvar contexto JSON para auditoria
with open(os.path.join(OUT, "pipeline_contexto.json"), "w", encoding="utf-8") as f:
    json.dump({
        "contexto":   ctx,
        "insights":   insights,
        "storyboard": {"big_idea": story["big_idea"],
                       "cta":      story["zona3_recomendacao"]["call_to_action"]},
    }, f, indent=2, ensure_ascii=False)

print(f"\n[OK] Dashboard HTML salvo em:\n  {out_html}")
print(f"\nBIG IDEA gerada pelo pipeline:")
print(f'  "{story["big_idea"]}"')

## 2.6 Visualizando o Protótipo no Notebook

O cell abaixo embute o HTML diretamente no notebook via `IPython.display`. Se a execução estiver em JupyterLab ou VS Code, o dashboard aparece inline. Caso contrário, abra o arquivo HTML no navegador.

In [ ]:
from IPython.display import IFrame, display

# Exibir inline (funciona melhor no JupyterLab)
display(IFrame(src=out_html, width="100%", height="700px"))

---
## Resumo dos Outputs Gerados

Todos os arquivos estão em `outputs/`:

In [ ]:
for f in sorted(os.listdir(OUT)):
    size = os.path.getsize(os.path.join(OUT, f))
    icon = "HTML" if f.endswith(".html") else ("CSV" if f.endswith(".csv") else
            "JSON" if f.endswith(".json") else "IMG")
    print(f"  [{icon}] {f:<45} {size/1024:>7.1f} KB")

---
## Próximos Passos — Versão com LLM Real

Para elevar este pipeline ao nível de produção, substitua cada função de agente por uma chamada a um LLM:

```python
# Exemplo com Amazon Bedrock (Claude)
import boto3, json

bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

def agente_insights_llm(ctx, exemplos_base):
    prompt = f"""
    Você é analista sênior de dados. Gere 2 insights acionáveis.
    
    CONTEXTO: {json.dumps(ctx)}
    EXEMPLOS DE QUALIDADE: {json.dumps(exemplos_base)}
    
    Retorne JSON com lista de insights no mesmo schema dos exemplos.
    """
    response = bedrock.invoke_model(
        modelId="anthropic.claude-3-5-sonnet-20241022-v2:0",
        body=json.dumps({"messages": [{"role": "user", "content": prompt}],
                         "max_tokens": 1024, "anthropic_version": "bedrock-2023-05-31"})
    )
    return json.loads(json.loads(response["body"].read())["content"][0]["text"])
```

Para orquestração, use **LangGraph** (local) ou **AWS Step Functions** (produção bancária com auditoria e retry automático).

Referências recomendadas:
- Yao et al. (2022) — ReAct: arXiv:2210.03629
- Wei et al. (2022) — Chain-of-Thought: arXiv:2201.11903  
- Hong et al. (2024) — Data Interpreter: arXiv:2402.18679
- AWS QuickSight Developer Guide: docs.aws.amazon.com/quicksight